# Feature Engineering

## Objective

Create forecasting features from the validated hourly appliance energy dataset.

The features are selected based on the findings from data understanding and exploratory data analysis.

### Feature groups

1. Calendar features
   - Hour of day
   - Day of week
   - Weekend indicator

2. Historical consumption features
   - Lag 1, 2, 3 hours
   - Lag 24 hours
   - Lag 48 hours
   - Lag 168 hours
   - Lag 672 hours (28 days)

3. Historical rolling features
   - 3-hour rolling mean
   - 6-hour rolling mean

### Important forecasting rule

Only information that would be available at the time of prediction can be used to create features.

Future target values and unavailable future sensor readings must not be used.

This prevents data leakage and ensures that the final forecasting system represents a realistic prediction scenario.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
hourly_df = pd.read_csv(
    "D:/Energy_Forecasting/data/processed/hourly_energy_data.csv",
    parse_dates=["date"]
)
hourly_df = hourly_df.set_index("date")

In [4]:
print("Shape:", hourly_df.shape)
print("First timestamp:", hourly_df.index.min())
print("Last timestamp:", hourly_df.index.max())

Shape: (3289, 26)
First timestamp: 2016-01-11 17:00:00
Last timestamp: 2016-05-27 17:00:00


In [5]:
forecast_df = hourly_df.copy()

print("Original shape:", hourly_df.shape)
print("Forecast dataframe shape:", forecast_df.shape)

Original shape: (3289, 26)
Forecast dataframe shape: (3289, 26)


In [6]:
forecast_df["hour"] = forecast_df.index.hour

In [7]:
#monday=0, sunday=6
forecast_df["day_of_week"] = forecast_df.index.dayofweek

In [8]:
#weekdays = 0, weekends= 1
forecast_df["is_weekend"] = (
    forecast_df["day_of_week"] >= 5
).astype(int)

In [9]:
forecast_df[
    ["hour", "day_of_week", "is_weekend"]
].head(10)

,hour,day_of_week,is_weekend
date,,,
2016-01-11 17:00:00,17,0,0
2016-01-11 18:00:00,18,0,0
2016-01-11 19:00:00,19,0,0
2016-01-11 20:00:00,20,0,0
2016-01-11 21:00:00,21,0,0
2016-01-11 22:00:00,22,0,0
2016-01-11 23:00:00,23,0,0
2016-01-12 00:00:00,0,1,0
2016-01-12 01:00:00,1,1,0


In [10]:
print(
    forecast_df[
        ["hour", "day_of_week", "is_weekend"]
    ].describe()
)

              hour  day_of_week   is_weekend
count  3289.000000  3289.000000  3289.000000
mean     11.501672     2.977197     0.277288
std       6.922851     1.985906     0.447728
min       0.000000     0.000000     0.000000
25%       6.000000     1.000000     0.000000
50%      12.000000     3.000000     0.000000
75%      17.000000     5.000000     1.000000
max      23.000000     6.000000     1.000000


In [11]:
print(
    forecast_df["day_of_week"].value_counts().sort_index()
)

day_of_week
0    463
1    480
2    480
3    480
4    474
5    456
6    456
Name: count, dtype: int64


In [12]:
lags = [1, 2, 3, 24, 48, 168, 672]

In [13]:
for lag in lags:
    forecast_df[f"lag_{lag}"] = (
        forecast_df["Appliances"].shift(lag)
    )

In [14]:
forecast_df[
    [
        "Appliances",
        "lag_1",
        "lag_2",
        "lag_3",
        "lag_24",
        "lag_48",
        "lag_168",
        "lag_672"
    ]
].head(10)

,Appliances,lag_1,lag_2,lag_3,lag_24,lag_48,lag_168,lag_672
date,,,,,,,,
2016-01-11 17:00:00,330,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-01-11 18:00:00,1060,330.0,NaN,NaN,NaN,NaN,NaN,NaN
2016-01-11 19:00:00,1040,1060.0,330.0,NaN,NaN,NaN,NaN,NaN
2016-01-11 20:00:00,750,1040.0,1060.0,330.0,NaN,NaN,NaN,NaN
2016-01-11 21:00:00,620,750.0,1040.0,1060.0,NaN,NaN,NaN,NaN
2016-01-11 22:00:00,1600,620.0,750.0,1040.0,NaN,NaN,NaN,NaN
2016-01-11 23:00:00,340,1600.0,620.0,750.0,NaN,NaN,NaN,NaN
2016-01-12 00:00:00,850,340.0,1600.0,620.0,NaN,NaN,NaN,NaN
2016-01-12 01:00:00,1010,850.0,340.0,1600.0,NaN,NaN,NaN,NaN


In [15]:
forecast_df[
    [
        "lag_1",
        "lag_2",
        "lag_3",
        "lag_24",
        "lag_48",
        "lag_168",
        "lag_672"
    ]
].isnull().sum()

lag_1        1
lag_2        2
lag_3        3
lag_24      24
lag_48      48
lag_168    168
lag_672    672
dtype: int64

In [16]:
forecast_df["rolling_mean_3"] = (
    forecast_df["Appliances"]
    .shift(1)
    .rolling(window=3)
    .mean()
)

In [17]:
forecast_df["rolling_mean_6"] = (
    forecast_df["Appliances"]
    .shift(1)
    .rolling(window=6)
    .mean()
)

In [18]:
forecast_df[
    [
        "Appliances",
        "rolling_mean_3",
        "rolling_mean_6"
    ]
].head(10)

,Appliances,rolling_mean_3,rolling_mean_6
date,,,
2016-01-11 17:00:00,330,NaN,NaN
2016-01-11 18:00:00,1060,NaN,NaN
2016-01-11 19:00:00,1040,NaN,NaN
2016-01-11 20:00:00,750,810.000000,NaN
2016-01-11 21:00:00,620,950.000000,NaN
2016-01-11 22:00:00,1600,803.333333,NaN
2016-01-11 23:00:00,340,990.000000,900.000000
2016-01-12 00:00:00,850,853.333333,901.666667
2016-01-12 01:00:00,1010,930.000000,866.666667


In [19]:
forecast_df[
    [
        "rolling_mean_3",
        "rolling_mean_6"
    ]
].isnull().sum()

rolling_mean_3    3
rolling_mean_6    6
dtype: int64

In [20]:
forecast_df.columns.tolist()

['Appliances',
 'lights',
 'T1',
 'RH_1',
 'T2',
 'RH_2',
 'T3',
 'RH_3',
 'T4',
 'RH_4',
 'T5',
 'RH_5',
 'T6',
 'RH_6',
 'T7',
 'RH_7',
 'T8',
 'RH_8',
 'T9',
 'RH_9',
 'T_out',
 'Press_mm_hg',
 'RH_out',
 'Windspeed',
 'Visibility',
 'Tdewpoint',
 'hour',
 'day_of_week',
 'is_weekend',
 'lag_1',
 'lag_2',
 'lag_3',
 'lag_24',
 'lag_48',
 'lag_168',
 'lag_672',
 'rolling_mean_3',
 'rolling_mean_6']

In [21]:
forecast_df.shape

(3289, 38)

In [22]:
forecast_df[
    [
        "Appliances",
        "rolling_mean_3",
        "rolling_mean_6"
    ]
].iloc[10:16]

,Appliances,rolling_mean_3,rolling_mean_6
date,,,
2016-01-12 03:00:00,210,710.000000,781.666667
2016-01-12 04:00:00,270,496.666667,713.333333
2016-01-12 05:00:00,260,250.000000,491.666667
2016-01-12 06:00:00,240,246.666667,478.333333
2016-01-12 07:00:00,340,256.666667,376.666667
2016-01-12 08:00:00,520,280.000000,265.000000


In [23]:
timestamp = forecast_df.index[5]

previous_3 = forecast_df["Appliances"].loc[
    timestamp - pd.Timedelta(hours=3):
    timestamp - pd.Timedelta(hours=1)
]

print(previous_3)
print("Manual mean:", previous_3.mean())
print(
    "Feature value:",
    forecast_df.loc[timestamp, "rolling_mean_3"]
)

date
2016-01-11 19:00:00    1040
2016-01-11 20:00:00     750
2016-01-11 21:00:00     620
Name: Appliances, dtype: int64
Manual mean: 803.3333333333334
Feature value: 803.3333333333334


In [24]:
missing_values = forecast_df.isnull().sum()

missing_values[missing_values > 0]

lag_1               1
lag_2               2
lag_3               3
lag_24             24
lag_48             48
lag_168           168
lag_672           672
rolling_mean_3      3
rolling_mean_6      6
dtype: int64

In [25]:
print("Total missing values:", forecast_df.isnull().sum().sum())

Total missing values: 927


In [26]:
feature_columns = [
    "hour",
    "day_of_week",
    "is_weekend",
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_24",
    "lag_48",
    "lag_168",
    "lag_672",
    "rolling_mean_3",
    "rolling_mean_6"
]

first_complete_timestamp = forecast_df[
    feature_columns
].dropna().index.min()

print("First timestamp with all engineered features available:")
print(first_complete_timestamp)

First timestamp with all engineered features available:
2016-02-08 17:00:00


### Excluding the 28-Day Lag Feature

The `lag_672` feature represents appliance energy consumption 28 days earlier (672 hours).

Although `lag_672` showed a moderate correlation with the target (0.322), using this feature would require 28 days of historical data before a prediction could be made. This would remove the first 672 observations from our relatively small dataset of 3,289 hourly observations.

The `lag_168` feature (one week earlier) showed a slightly stronger correlation (0.329) while requiring only 7 days of historical data.

Therefore, `lag_672` is excluded to preserve more training data while retaining a useful weekly historical pattern through `lag_168`.

In [27]:
forecast_df = forecast_df.drop(columns=["lag_672"])

In [28]:
feature_columns = [
    "hour",
    "day_of_week",
    "is_weekend",
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_24",
    "lag_48",
    "lag_168",
    "rolling_mean_3",
    "rolling_mean_6"
]

In [29]:
missing_values = forecast_df.isnull().sum()

missing_values[missing_values > 0]

lag_1               1
lag_2               2
lag_3               3
lag_24             24
lag_48             48
lag_168           168
rolling_mean_3      3
rolling_mean_6      6
dtype: int64

In [30]:
first_complete_timestamp = forecast_df[
    feature_columns
].dropna().index.min()

print(
    "First timestamp with all engineered features available:"
)
print(first_complete_timestamp)

First timestamp with all engineered features available:
2016-01-18 17:00:00


In [31]:
print("Shape:", forecast_df.shape)

Shape: (3289, 37)


In [32]:
forecast_df = forecast_df.dropna(
    subset=feature_columns
).copy()

In [33]:
print("Shape:", forecast_df.shape)
print("First timestamp:", forecast_df.index.min())
print("Last timestamp:", forecast_df.index.max())

Shape: (3121, 37)
First timestamp: 2016-01-18 17:00:00
Last timestamp: 2016-05-27 17:00:00


In [34]:
print(
    "Total missing values:",
    forecast_df.isnull().sum().sum()
)

Total missing values: 0


In [35]:
forecast_df.isnull().sum()[
    forecast_df.isnull().sum() > 0
]

Series([], dtype: int64)

In [36]:
print(
    "Chronologically ordered:",
    forecast_df.index.is_monotonic_increasing
)

Chronologically ordered: True


In [37]:
print(
    "Duplicate timestamps:",
    forecast_df.index.duplicated().sum()
)

Duplicate timestamps: 0


In [38]:
forecast_df.to_csv(
    "D:/Energy_Forecasting/data/processed/engineered_energy_data.csv"
)

In [39]:
test_df = pd.read_csv(
    "D:/Energy_Forecasting/data/processed/engineered_energy_data.csv",
    parse_dates=["date"]
)

print("Saved dataset shape:", test_df.shape)
print("Saved dataset columns:", len(test_df.columns))

Saved dataset shape: (3121, 38)
Saved dataset columns: 38
